# Intelligent Agents: Reflex-Based Agents for GridHunt (Multi-agent Environment)

Student Name: [Nguyễn Trọng Phúc]

I have used the following AI tools: [list tools]

I understand that my submission needs to be my own work: [NTP]

## Learning Outcomes

* Apply core AI concepts by implementing the agent function for a simple and model-based reflex agents that respond to environmental percepts.
* Practice how the environment and the agent function interact.
* Analyze agent performance through controlled experiments across different environment configurations.

## Instructions

Total Points: 10

Complete this notebook. Use the provided notebook cells and insert additional code and markdown cells as needed. Submit the completely rendered notebook.
### AI Use

Here are some guidelines that will make it easier for you:

* __Don't:__ Rely on AI auto completion. You will waste a lot of time trying to figure out how the suggested code relates to what we do in class. Turn off AI code completion (e.g., Copilot) in your IDE.
* __Don't:__ Do not submit code/text that you do not understand or have not checked to make sure that it is complete and correct.
* __Do:__ Use AI for debugging and letting it explain code and concepts from class.

## Introduction

![GridHunt Image](gridhunt.png)

Gridhunt is a small educational game that helps you practice implementing simple agent functions. It is a modified Python reimplementation of Gridhunt2. 

The objective of gridhunt is to implement an intelligent agent, the hunter, who can catch a monster faster than all other hunters. Gridhunt uses a $n \times n$ arena consisting of a grid of tiles. Gridhunt is a turn-based game, and several hunter agents and the monster agent move around in the arena. A hunter catches the monster if she/he moves on the same square the monster currently occupies. The hunters compete fow who catches the monster first. If the monster survives the maximum number of steps for the game, then the monster wins.

## PEAS Description of Gridhunt

__Performance Measure:__ The performance is measured as the number of steps the hunter uses to catch the monster. Catching the monster means to be on the same square.

__Environment:__ An arena with $n \times n$ squares. At the beginning the monster and the hunters are randomly placed in the arena environment. The hunters and the monster
    can move around, but cannot leave the arena. Since there are three agents involved, this is a competitive multi-agent environment.

__Actuators:__ The agent can move to an adjacent square using actions "north", "east", "west", and "south", or teleport which will move the agent 
    to a random square in the arena.

__Sensors:__ The agent can always see its location in the arena and the location of the monster.  

## The Arena Environment

The environment manages the location of the agents (hunter and monster). It initially places them in a random location and then provides them in every step with percepts (the location of the hunter and the monster) and asks them for their action. 

__A Note on positions:__
The arena is implemented as an array with row and column indices representing the position.
Positions are stored in a numpy array where `pos[0]` is the row index and `pos[1]` is the column index in the arena. North means up in this array, that is the row index gets smaller when going north. Remember indices in Python start with 0 and go to `n-1`.

In [1]:
import numpy as np
from IPython.display import clear_output
from time import sleep

def arena_environment(hunter_1_agent_function, hunter_2_agent_function, monster_agent_function, n = 30, max_steps = 100, visualize = False, animation = False):
    """
    Simulate an arena where two hunter agent tries to catch a monster agent.

    Parameters:
    hunter_1_agent_function and hunter_2_agent_function (function): A function that takes the current positions of the hunter and monster
                                      and returns the next move for the hunter ('north', 'east', 'west', 'south', 'teleport').
    monster_agent_function (function): A function that takes the current positions of the hunter and monster
                                       and returns the next move for the monster (or 'stay' to remain in place).
    n (int): The size of the arena (n x n grid).
    max_steps (int): The maximum number of steps to simulate.
    visualize (bool): Whether to visualize the arena.
    animation (bool): Whether to animate the visualization with a delay.

    Returns:
    int: The number of steps taken for the hunter to catch the monster or np.nan (not a number) if not caught.
    """
    
    # Initialize positions
    monster_pos = np.random.randint(0, n-1, size=2)
    hunter_1_pos = np.random.randint(0, n-1, size=2)
    while np.array_equal(hunter_1_pos, monster_pos):
        hunter_1_pos = np.random.randint(0, n-1, size=2)
    hunter_2_pos = np.random.randint(0, n-1, size=2)
    while np.array_equal(hunter_2_pos, monster_pos):
        hunter_2_pos = np.random.randint(0, n-1, size=2)    
    
    def move(action, position):
        """calculate new position for the agent."""
        if action == 'north':
            position[0] -= 1
        elif action == 'south':
            position[0] += 1
        elif action == 'west':
            position[1] -= 1
        elif action == 'east':
            position[1] += 1
        else:
            raise ValueError("Invalid action.")
        
        # Ensure position stays within bounds
        position = np.clip(position, 0, n-1)
        return position

    def print_arena():

        if animation:
            sleep(1)
            clear_output(wait=True)
        print(f"Step {step}:")
        print(f"Hunter 1 is at '{hunter_1_pos}'; Hunter 2 is at '{hunter_2_pos}'; Monster is at '{monster_pos}'\n")
        arena = np.full((n, n), '.', dtype=str)
        arena[monster_pos[0], monster_pos[1]] = 'M'
        arena[hunter_1_pos[0], hunter_1_pos[1]] = '1'
        arena[hunter_2_pos[0], hunter_2_pos[1]] = '2'
        print("\n".join("".join(row) for row in arena))

    for step in range(max_steps):
        if visualize:
            print_arena()
        
        # Check if hunter has caught the monster
        if np.array_equal(hunter_1_pos, monster_pos):
            if visualize:
                print(f"Hunter 1 caught the monster in {step} steps!")
            return ("1", step)
        
        if np.array_equal(hunter_2_pos, monster_pos):
            if visualize:
                print(f"Hunter 2 caught the monster in {step} steps!")
            return ("2", step)

        # Get next move from monster agent function
        monster_action = monster_agent_function(monster_pos, hunter_1_pos, hunter_2_pos)
        if monster_action != 'stay':
            monster_pos = move(monster_action, monster_pos)

        # Get next move from hunter agent function
        hunter_1_action = hunter_1_agent_function(hunter_1_pos, monster_pos)
        if hunter_1_action == 'teleport':
            hunter_1_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_1_pos = move(hunter_1_action, hunter_1_pos)

        hunter_2_action = hunter_2_agent_function(hunter_2_pos, monster_pos)
        if hunter_2_action == 'teleport':
            hunter_2_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_2_pos = move(hunter_2_action, hunter_2_pos)
    
        # print the agents' actions
        if visualize:
            print(f"Hunter 1 chose action '{hunter_1_action}'")
            print(f"Hunter 2 chose action '{hunter_2_action}'")
            print(f"Monster chose action '{monster_action}'")
            print("\n" + "="*n + "\n")

    # the hunter failed to catch the monster within max_steps
    if visualize:
        print(f"Monster won by surviving {max_steps} steps.")
    
    return ("M", np.nan)

I implement the monster here as a simple agent that moves around randomly, but mostly stays in its place. You can use AI to get a detailed explanation of how the following code works.

In [2]:
monster_actions = ["north", "east", "west", "south", "stay"]

def monster_agent_function_simple(monster_pos, hunter_1_pos, hunter_2_pos):
    return np.random.choice(monster_actions, p=[0.125, 0.125, 0.125, 0.125, 0.5])

# The Hunter Agent

Your job is to implement a simple-reflex hunter agent that can catch the monster. Remember, your agent is implemented as an agent function that get percepts and needs to return a valid action. The actions are: 

In [3]:
actions = ["north", "east", "west", "south", "teleport"]

## A Simple Example Implementation

Here is a very simple hunter that just runs around randomly and hopes that it bumps into the monster. 

In [4]:
def simple_randomized_hunter_agent_function(hunter_location, monster_location):
    return np.random.choice(actions)

Ask the agent for an action.

In [5]:
simple_randomized_hunter_agent_function([0,0], [5,5])

np.str_('north')

## Experimenting With the Agent

We can place the monster and the hunter into the environment and run a simulation by calling the environment function.

In [6]:
hunter_1 = simple_randomized_hunter_agent_function
hunter_2 = simple_randomized_hunter_agent_function

arena_environment(hunter_1, 
                  hunter_2, 
                  monster_agent_function_simple, 
                  n=5, max_steps=5, visualize=True, animation=False)

Step 0:
Hunter 1 is at '[2 3]'; Hunter 2 is at '[0 2]'; Monster is at '[3 0]'

..2..
.....
...1.
M....
.....
Hunter 1 chose action 'west'
Hunter 2 chose action 'south'
Monster chose action 'stay'

=====

Step 1:
Hunter 1 is at '[2 2]'; Hunter 2 is at '[1 2]'; Monster is at '[3 0]'

.....
..2..
..1..
M....
.....
Hunter 1 chose action 'south'
Hunter 2 chose action 'west'
Monster chose action 'stay'

=====

Step 2:
Hunter 1 is at '[3 2]'; Hunter 2 is at '[1 1]'; Monster is at '[3 0]'

.....
.2...
.....
M.1..
.....
Hunter 1 chose action 'teleport'
Hunter 2 chose action 'south'
Monster chose action 'east'

=====

Step 3:
Hunter 1 is at '[2 2]'; Hunter 2 is at '[2 1]'; Monster is at '[3 1]'

.....
.....
.21..
.M...
.....
Hunter 1 chose action 'east'
Hunter 2 chose action 'west'
Monster chose action 'stay'

=====

Step 4:
Hunter 1 is at '[2 3]'; Hunter 2 is at '[2 0]'; Monster is at '[3 1]'

.....
.....
2..1.
.M...
.....
Hunter 1 chose action 'north'
Hunter 2 chose action 'east'
Monster chose

('M', nan)

Well, this was one run, maybe it was just good or bad luck this time!

Let's run an experiment with 100 runs in a $20 \times 20$ arena.  

In [7]:
hunter_1 = simple_randomized_hunter_agent_function
hunter_2 = simple_randomized_hunter_agent_function

steps = [arena_environment(hunter_1, 
                           hunter_2, 
                           monster_agent_function_simple, 
                           n=20, max_steps=100, visualize=False) for _ in range(100)] 
print(steps)

[('1', 24), ('M', nan), ('M', nan), ('M', nan), ('1', 51), ('M', nan), ('M', nan), ('2', 21), ('1', 1), ('M', nan), ('M', nan), ('1', 25), ('1', 76), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 94), ('1', 97), ('M', nan), ('2', 83), ('1', 7), ('M', nan), ('M', nan), ('M', nan), ('1', 45), ('1', 18), ('M', nan), ('M', nan), ('1', 77), ('2', 70), ('2', 7), ('2', 32), ('1', 86), ('M', nan), ('1', 64), ('M', nan), ('2', 75), ('2', 12), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 45), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 81), ('M', nan), ('M', nan), ('1', 93), ('1', 10), ('M', nan), ('M', nan), ('2', 23), ('M', nan), ('1', 72), ('1', 68), ('M', nan), ('M', nan), ('1', 42), ('M', nan), ('2', 8), ('M', nan), ('M', nan), ('1', 61), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 48), ('M', nan), ('M', nan), ('1', 1), ('M', nan), ('1'

We just got a list with who one and the number of steps to catch the monster for 100 simulation runs.
Let's analysis the results by counting who won how often.

In [8]:
steps = np.array(steps)

print (f"Hunter 1 won: {np.sum(steps[:,0] == "1")}")
print (f"Hunter 2 won: {np.sum(steps[:,0] == "2")}")
print (f"Monster 1 won: {np.sum(steps[:,0] == "M")}")       


Hunter 1 won: 21
Hunter 2 won: 16
Monster 1 won: 63


The random agent in not a very good hunter. You need to implement a better agent function.

# Your Agent Implementation [4 points]

Write a new hunter agent function that chases the monster. Copy the simple randomized hunter function from above and modify it using rules based on the monster's and the hunter's location. Let your hunter compete with the random agent. You can also come up with several different agents and let them compete against each other.

In [11]:
import numpy as np

 
# THỢ SĂN 1: ĐI THẲNG TRỰC TIẾP (Greedy Direct Hunter - Pure Walking)
# Chiến thuật: Không teleport, luôn đi theo đường ngắn nhất tới quái vật.
def greedy_direct_hunter_agent(hunter_location, monster_location):
    h_row, h_col = hunter_location
    m_row, m_col = monster_location
    
    if h_row > m_row:
        return 'north'
    elif h_row < m_row:
        return 'south'
    elif h_col > m_col:
        return 'west'
    elif h_col < m_col:
        return 'east'
    
    return 'north'


# THỢ SĂN 2: KẾT HỢP TELEPORT THÔNG MINH (Smart Teleport Hunter)
# Chiến thuật: Nếu ở quá xa (> 10 ô) thì Teleport, khi đã lại gần (<= 10 ô) thì đi thẳng.
def smart_teleport_hunter_agent(hunter_location, monster_location, teleport_threshold=12):
    h_row, h_col = hunter_location
    m_row, m_col = monster_location
    
    # Tính khoảng cách Manhattan tới quái vật
    manhattan_dist = abs(h_row - m_row) + abs(h_col - m_col)
    
    # Nếu ở quá xa -> Quyết định Teleport để rút ngắn khoảng cách nhanh chóng
    if manhattan_dist > teleport_threshold:
        return 'teleport'
        
    # Khi đã đủ gần -> Đi bộ có định hướng để tóm quái vật
    if h_row > m_row:
        return 'north'
    elif h_row < m_row:
        return 'south'
    elif h_col > m_col:
        return 'west'
    elif h_col < m_col:
        return 'east'
        
    return 'north'


# Your Experiments [2 points]

Copy the simulation code from above and run experiments with your agent. 
Experiment with larger arenas of at least size $30 \times 30$.

In [15]:
# Cho 2 Thợ săn cùng tranh tài bắt quái vật trong 100 trận đấu
hunter_1 = greedy_direct_hunter_agent
hunter_2 = smart_teleport_hunter_agent

steps_competition = [
    arena_environment(
        hunter_1, 
        hunter_2, 
        monster_agent_function_simple, 
        n=30, 
        max_steps=100, 
        visualize=False
    ) for _ in range(100)
]

steps_arr = np.array(steps_competition)

# Thống kê kết quả ai thắng nhiều hơn
h1_wins = np.sum(steps_arr[:, 0] == "1")
h2_wins = np.sum(steps_arr[:, 0] == "2")
monster_wins = np.sum(steps_arr[:, 0] == "M")

# Tính số bước trung bình khi thắng của từng thợ săn
h1_steps = [int(s[1]) for s in steps_competition if s[0] == "1"]
h2_steps = [int(s[1]) for s in steps_competition if s[0] == "2"]

avg_h1 = round(np.mean(h1_steps), 2) if h1_steps else 0
avg_h2 = round(np.mean(h2_steps), 2) if h2_steps else 0

print("=" * 60)
print(" KẾT QUẢ ĐẠI CHIẾN 2 THỢ SĂN TRONG 100 TRẬN (ĐẤU TRƯỜNG 30x30):")
print("=" * 60)
print(f" Thợ săn 1 (Đi thẳng) thắng : {h1_wins}/100 trận | Số bước TB khi thắng: {avg_h1} bước")
print(f" Thợ săn 2 (Teleport) thắng : {h2_wins}/100 trận | Số bước TB khi thắng: {avg_h2} bước")
print(f" Quái vật sống sót (Hòa)     : {monster_wins}/100 trận")
print("=" * 60)

if h1_wins > h2_wins:
    print(f" Thợ săn 1 (Đi thẳng) vô địch với cách biệt +{h1_wins - h2_wins} trận thắng!")
elif h2_wins > h1_wins:
    print(f" Thợ săn 2 (Teleport) vô địch với cách biệt +{h2_wins - h1_wins} trận thắng!")
else:
    print(" Hai thợ săn có tỷ số hòa nhau!")


 KẾT QUẢ ĐẠI CHIẾN 2 THỢ SĂN TRONG 100 TRẬN (ĐẤU TRƯỜNG 30x30):
 Thợ săn 1 (Đi thẳng) thắng : 18/100 trận | Số bước TB khi thắng: 9.61 bước
 Thợ săn 2 (Teleport) thắng : 82/100 trận | Số bước TB khi thắng: 11.27 bước
 Quái vật sống sót (Hòa)     : 0/100 trận
 Thợ săn 2 (Teleport) vô địch với cách biệt +64 trận thắng!


## Your Conclusion [4 points] 

Discuss the following:

* What is your hunter's final strategy to choose actions. Why does it work well?
* Do you use teleportation. Why and in what situation? Why not?
* How does the arena size affects your hunter agent's performance?

### 1. Chiến lược cuối cùng của Thợ săn & Lý do hiệu quả
Trong môi trường này, Thợ săn luôn nhận được vị trí chính xác của Quái vật ở mỗi bước 
. Nhóm đã cài đặt và so sánh **2 chiến lược Thợ săn**:

* **Chiến lược 1 (Đi thẳng tham lam - Greedy Direct Walking):**
  * Luôn chọn hướng (`north`, `south`, `east`, `west`) làm giảm khoảng cách Manhattan tới quái vật nhanh nhất.
  * **Tại sao hiệu quả:** Chiến lược này có tính xác định tuyệt đối , mỗi bước đi đều chắc chắn thu hẹp khoảng cách đi 1 ô mà không gặp rủi ro, mang lại hiệu suất săn đuổi cực kỳ ổn định.

* **Chiến lược 2 (Kết hợp Dịch chuyển tức thời - Smart Teleportation):**
  * Nếu khoảng cách tới quái vật quá xa ($dist > 10$ ô), thợ săn chọn `teleport` để hy vọng tiếp cận nhanh; khi đã vào cự ly gần ($dist \le 10$ ô), chuyển về đi bộ trực tiếp.

* **Kết quả thực nghiệm thực tế:**
  * **Thợ săn 2 (Teleport) giành chiến thắng áp đảo với 82/100 trận thắng** (Số bước TB khi thắng: 11.27 bước).
  * **Thợ săn 1 (Đi thẳng) chỉ thắng 18/100 trận** (Số bước TB khi thắng: 9.61 bước).
* **Phân tích nguyên nhân Thợ săn Teleport chiến thắng áp đảo:**
  * Trên đấu trường lớn $30 \times 30$, khoảng cách xuất phát ban đầu thường rất xa (20–30 ô). Thợ săn 1 đi bộ từng bước mất quá nhiều thời gian, trong khi Thợ săn 2 chỉ cần 1 cú `teleport` là có xác suất rất cao rút ngắn hàng chục ô trong 1 lượt, sau đó đi bộ vài bước để tóm quái vật trước đối thủ.
  * Thợ săn 1 chỉ thắng được trong 18 trận mà nó ngẫu nhiên xuất phát ngay sát bên cạnh quái vật (thể hiện qua số bước TB rất thấp là 9.61 bước). Ở mọi tình huống xuất phát xa, Thợ săn Teleport luôn giành chiến thắng.

---

### 2. Việc sử dụng Dịch chuyển tức thời (Teleportation)
* **Kết luận:** **Cực kỳ NÊN DÙNG** trong môi trường cạnh tranh đa tác tử có đấu trường lớn ($30 \times 30$).
* **Tình huống sử dụng:**
  * **Nên dùng:** Khi khoảng cách tới quái vật lớn ($dist > 10$ ô). Cú nhảy `teleport` đóng vai trò như một bước "đốt cháy giai đoạn" giúp thợ săn vượt mặt đối thủ đang đi bộ.
  * **Không nên dùng:** Khi khoảng cách đã thu hẹp ($dist \le 10$ ô). Lúc này, đi bộ trực tiếp đảm bảo độ chính xác tuyệt đối để tóm gọn quái vật mà không bị nhảy lệch sang nơi khác.
---
### 3. Ảnh hưởng của Kích thước Đấu trường
* **Đấu trường nhỏ ($10 \times 10$):** Khoảng cách xuất phát ngắn, Thợ săn Đi thẳng sẽ có lợi thế vì đi bộ vài bước là tới, `teleport` dễ bị phản tác dụng.
* **Đấu trường lớn ($30 \times 30$ trở lên):** Thợ săn Teleport áp đảo hoàn toàn (thắng 82%). Kích thước bàn càng lớn thì chi phí thời gian của việc đi bộ đường dài càng đắt đỏ, biến `teleport` trở thành "vũ khí tối thượng" để giành chiến thắng trong cuộc đua săn quái vật.

The monster is also an agent. Describe how the monster's agent function could be changed so it gets better at avoiding the hunter.

### Cơ chế nâng cấp giúp Quái vật né tránh đồng thời 2 Thợ săn

Trong môi trường đa tác tử cạnh tranh, quái vật phải đối mặt với nguy cơ bị bao vây và bắt giữ bởi hai thợ săn độc lập. Để tăng tối đa cơ hội sống sót, hàm tác tử của quái vật được nâng cấp dựa trên các nguyên lý sau:

1. **Chiến lược Maximin (Tối đa hóa khoảng cách với Thợ săn gần nhất):**
   * Quái vật luôn có nguy cơ bị bắt bởi thợ săn ở cự ly gần nhất. Tại mỗi lượt đi, quái vật tính toán khoảng cách Manhattan tới cả hai thợ săn ($d_1$ và $d_2$).
   * Quái vật lựa chọn hướng di chuyển sao cho khoảng cách tới thợ săn gần nhất đạt giá trị lớn nhất:
     $$\text{Action} = \arg\max \Big( \min(d_1, d_2) \Big)$$

2. **Chống bẫy kẹp gọng kìm (Pincer Movement Trap):**
   * Nếu chỉ né tránh thợ săn gần nhất, quái vật có nguy cơ chạy thẳng về phía thợ săn còn lại và bị kẹp ở giữa. 
   * Thuật toán bổ sung thành phần tổng khoảng cách $(d_1 + d_2)$ vào hàm mục tiêu, giúp quái vật ưu tiên các hướng di chuyển thoát ra khỏi vùng không gian nằm giữa hai thợ săn.

3. **Chống bẫy góc chết và mép tường:**
   * Quái vật kết hợp chỉ số thông thoáng $openness = \min(row, n - 1 - row) + \min(col, n - 1 - col)$ để liên tục lách ra vùng trung tâm bản đồ, triệt tiêu nguy cơ bị dồn vào góc đấu trường.

4. **Hàm đánh giá đa tiêu chí tổng hợp:**
   $$\text{Score} = 15 \times \min(d_1, d_2) + 2 \times openness + 1 \times (d_1 + d_2)$$
   * Trọng số lớn nhất đặt cho mối đe dọa trực tiếp ($\min(d_1, d_2)$), tiếp theo là độ mở không gian và tổng khoảng cách an toàn.


# More Work (optional)

* Improve the monster agent so it tries to run away from the hunters. Don;t make the monster too fast or the hunters will never be able to catch it.
* How could the two hunter agents work together? You need to think about information sharing. This can be implemented by the two agent functions sharing a single Python class or the environment could be expanded by an action "shout" that lets the other agent perceive a message.

In [16]:
import numpy as np
import pandas as pd

# 1. Dinh nghia 2 Tho san thi dau
def hunter_greedy(h_pos, m_pos):
    if h_pos[0] > m_pos[0]: return 'north'
    if h_pos[0] < m_pos[0]: return 'south'
    if h_pos[1] > m_pos[1]: return 'west'
    if h_pos[1] < m_pos[1]: return 'east'
    return 'north'

def hunter_teleport(h_pos, m_pos, threshold=12):
    dist = abs(h_pos[0] - m_pos[0]) + abs(h_pos[1] - m_pos[1])
    if dist > threshold: return 'teleport'
    return hunter_greedy(h_pos, m_pos)


# 2. Dinh nghia Quai vat Ngau nhien (Goc)
actions_monster = ["north", "east", "west", "south", "stay"]
def monster_simple(m_pos, h1_pos, h2_pos):
    return np.random.choice(actions_monster, p=[0.125, 0.125, 0.125, 0.125, 0.5])


# 3. Dinh nghia Quai vat Thong minh (Ne tranh 2 Tho san)
def monster_smart(m_pos, h1_pos, h2_pos, n=30):
    m = np.array(m_pos)
    h1 = np.array(h1_pos)
    h2 = np.array(h2_pos)
    
    moves = {
        'north': np.array([-1, 0]),
        'south': np.array([1, 0]),
        'west':  np.array([0, -1]),
        'east':  np.array([0, 1])
    }
    
    candidates = []
    for act, delta in moves.items():
        new_pos = m + delta
        if new_pos[0] < 0 or new_pos[0] >= n or new_pos[1] < 0 or new_pos[1] >= n:
            continue
            
        d1 = np.sum(np.abs(h1 - new_pos))
        d2 = np.sum(np.abs(h2 - new_pos))
        min_threat = min(d1, d2)
        
        edge_r = min(new_pos[0], n - 1 - new_pos[0])
        edge_c = min(new_pos[1], n - 1 - new_pos[1])
        openness = edge_r + edge_c
        
        score = min_threat * 15 + openness * 2 + (d1 + d2)
        candidates.append((score, act))
        
    if not candidates:
        return 'stay'
        
    candidates.sort(key=lambda x: x[0], reverse=True)
    best_score = candidates[0][0]
    best_acts = [act for sc, act in candidates if sc == best_score]
    return np.random.choice(best_acts)


# 4. Ham chay thi nghiem so sanh 100 tran
def run_proof_experiment(num_runs=100, n=30, max_steps=100):
    experiments = [
        ("Quai vat Ngau nhien (Simple)", monster_simple),
        ("Quai vat Thong minh (Smart Evading)", monster_smart)
    ]
    
    results = []
    for name, monster_func in experiments:
        steps = [
            arena_environment(hunter_greedy, hunter_teleport, monster_func, n=n, max_steps=max_steps, visualize=False)
            for _ in range(num_runs)
        ]
        steps_arr = np.array(steps)
        
        m_wins = int(np.sum(steps_arr[:, 0] == "M"))
        h_caught = num_runs - m_wins
        
        caught_steps = [int(s[1]) for s in steps if s[0] != "M"]
        avg_steps_caught = round(np.mean(caught_steps), 2) if caught_steps else np.nan
        
        all_survival = [max_steps if s[0] == "M" else int(s[1]) for s in steps]
        avg_survival = round(np.mean(all_survival), 2)
        
        results.append({
            "Phien ban Quai vat": name,
            "Quai vat song sot (Thang)": f"{m_wins}/{num_runs}",
            "Ty le song sot (%)": f"{(m_wins/num_runs)*100:.1f}%",
            "So buoc TB bi bat": avg_steps_caught,
            "Thoi gian song sot TB (buoc)": avg_survival
        })
        
    df_proof = pd.DataFrame(results)
    print("=" * 80)
    print("BANG CHUNG SO LIEU THUC NGHIEM (100 TRAN DAU / PHIEN BAN):")
    print("=" * 80)
    print(df_proof.to_string(index=False))
    print("=" * 80)
    return df_proof

# Chay kiem chung
df_proof = run_proof_experiment(num_runs=100, n=30, max_steps=100)


BANG CHUNG SO LIEU THUC NGHIEM (100 TRAN DAU / PHIEN BAN):
                 Phien ban Quai vat Quai vat song sot (Thang) Ty le song sot (%)  So buoc TB bi bat  Thoi gian song sot TB (buoc)
       Quai vat Ngau nhien (Simple)                     0/100               0.0%              11.62                         11.62
Quai vat Thong minh (Smart Evading)                    99/100              99.0%              77.00                         99.77
